In [1]:
# Install libraries
!pip install rasterio
!pip install geopandas
# Load the data from the repository
!curl -L https://github.com/a-taylor1/DATA450_Team5/raw/f96e6ea78ed178af4fb4406b0d911c6bb98e1e0b/test_data.zip --output test_data.zip
!unzip -qo test_data.zip
%cd LF2024_EVC_CONUS/
# Print all of the files in the folder
!ls -l

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2814k  100 2814k    0     0  3518k      0 --:--:-- --:--:-- --:--:-- 8158k
/content/LF2024_EVC_CONUS
total 9560
-rwxrwxrwx 1 root root  103426 Apr  7 00:53 LF2024_EVC_CONUS.dbf
-rwxrwxrwx 1 root root     337 Apr  7 00:53 LF2024_EVC_CONUS.GeoJSON
-rwxrwxrwx 1 root root      93 Apr  7 00:53 LF2024_EVC_CONUS.tfw
-rwxrwxrwx 1 root root 6726706 Apr  7 00:53 LF2024_EVC_CONUS.tif
-rwxrwxrwx 1 root root 2819920 Apr  7 00:53 LF2024_EVC_CONUS.tif.ovr
-rwxrwxrwx 1 root root   62224 Apr  7 00:53 LF2024_EVC_CONUS.tif.vat.dbf
-rwxrwxrwx 1 root root    1622 Apr  7 00:53 metadata.dbf
-rwxrwxrwx 1 root root     459 Apr  7 00:53 metadata.prj
-rwxrwxrwx 1 root root    2132 Apr  7 00:53 metadata.shp
-rwxrwxrwx 1 root root     116 Apr  7 00:53 metadata.shx
-rwxrwxrwx 1 

In [2]:
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from geopandas import read_file

%matplotlib inline

In [3]:
# IMPORT DATAFRAME OF EV CODES
key_table:pd.DataFrame = read_file('LF2024_EVC_CONUS.dbf')
key_table = key_table.drop(columns = ['R', 'G', 'B', 'RED', 'GREEN', 'BLUE'])
key_table

,VALUE,CLASSNAMES
0,-9999,Fill-NoData
1,11,Open Water
2,12,Snow/Ice
3,13,Developed-Upland Deciduous Forest
4,14,Developed-Upland Evergreen Forest
...,...,...
288,395,Herb Cover = 95%
289,396,Herb Cover = 96%
290,397,Herb Cover = 97%
291,398,Herb Cover = 98%


### **SORT CLASSNMES INTO UNIQUE COLUMNS**

In [4]:
# 1. Get all levels of CLASSNAMES
class_levels = key_table['CLASSNAMES'].to_list()
class_levels[1:10]

['Open Water',
 'Snow/Ice',
 'Developed-Upland Deciduous Forest',
 'Developed-Upland Evergreen Forest',
 'Developed-Upland Mixed Forest',
 'Developed-Upland Herbaceous',
 'Developed-Upland Shrubland',
 'Developed - Open Space',
 'Developed-Low Intensity']

In [5]:
# 2. Separate continuous and discrete classes
#   NOTE: The dataframe represents continuous classes in the form "class name = xx%"
continuous_classes = []
discrete_classes = []
for c in class_levels:
  try:
    c.index('=')
    continuous_classes.append(c)
  except:
    discrete_classes.append(c)
print(f'CONTINUOUS:\n{continuous_classes[1:10]}')
print(f'CONTINUOUS:\n{discrete_classes[1:10]}')

CONTINUOUS:
['Tree Cover = 11%', 'Tree Cover = 12%', 'Tree Cover = 13%', 'Tree Cover = 14%', 'Tree Cover = 15%', 'Tree Cover = 16%', 'Tree Cover = 17%', 'Tree Cover = 18%', 'Tree Cover = 19%']
CONTINUOUS:
['Open Water', 'Snow/Ice', 'Developed-Upland Deciduous Forest', 'Developed-Upland Evergreen Forest', 'Developed-Upland Mixed Forest', 'Developed-Upland Herbaceous', 'Developed-Upland Shrubland', 'Developed - Open Space', 'Developed-Low Intensity']


In [6]:
continuous_formatted = []
discrete_formatted = []
# 3. Create columns for continuous classes
def translate_classname(name:str) -> tuple[str, str]:
  split_c = [name]
  if name in continuous_classes:
    if ' = ' in name: split_c = name.split(' = ')
    elif ' >= ' in name: split_c = name.split(' >= ')
    split_c[0] = split_c[0].replace(' ', '_')
    split_c[1] = int(split_c[1].replace('%', ''))
    continuous_formatted.append(split_c[0])
  else:
    split_c[0] = name.replace(' ', '_')
    split_c.append(True)
    discrete_formatted.append(split_c[0])
  return tuple(split_c)
def extract_classname(name:str) -> str:
  return translate_classname(name)[0].lower()
def extract_value(name:str) -> int|bool:
  return translate_classname(name)[1]

key_table['class_name'] = key_table['CLASSNAMES'].apply(extract_classname).rename('class_name')
key_table['class_value'] = key_table['CLASSNAMES'].apply(extract_value).rename('class_value')
key_table

,VALUE,CLASSNAMES,class_name,class_value
0,-9999,Fill-NoData,fill-nodata,True
1,11,Open Water,open_water,True
2,12,Snow/Ice,snow/ice,True
3,13,Developed-Upland Deciduous Forest,developed-upland_deciduous_forest,True
4,14,Developed-Upland Evergreen Forest,developed-upland_evergreen_forest,True
...,...,...,...,...
288,395,Herb Cover = 95%,herb_cover,95
289,396,Herb Cover = 96%,herb_cover,96
290,397,Herb Cover = 97%,herb_cover,97
291,398,Herb Cover = 98%,herb_cover,98


In [7]:
import rasterio
import numpy as np
import pandas as pd

# 1. First, create map_df by reading the raster
with rasterio.open('LF2024_EVC_CONUS.tif') as src:
    band = src.read(1)
    transform = src.transform
    cols, rows = np.meshgrid(np.arange(src.width), np.arange(src.height))
    xs, ys = transform * (cols, rows)

map_df = pd.DataFrame({
    'x': xs.flatten(),
    'y': ys.flatten(),
    'VALUE': band.flatten()
})

# 2. Check the raster's official NoData value
with rasterio.open('LF2024_EVC_CONUS.tif') as src:
    print(f"Raster NoData value: {src.nodata}")

# 3. Find unique values in our map dataframe
unique_map_values = map_df['VALUE'].unique()
print(f"Unique values in map: {unique_map_values}")

# 4. Find which values are missing from the key_table
missing_from_key = set(unique_map_values) - set(key_table['VALUE'])
print(f"Values in map but missing from key_table: {missing_from_key}")

# 5. Map the raster's NoData value (32767) to the key_table's NoData value (-9999)
map_df['VALUE'] = map_df['VALUE'].replace(32767, -9999)
print("\nReplaced 32767 with -9999 to match key_table.")
display(map_df.head())


Raster NoData value: 32767.0
Unique values in map: [32767   332   349   348   346   343   340   344   358   342   347   345
   351   355   338   339   341   232   337   352   350   228   225   224
   353   334   330   335   229   227   336   234   362   359   329   357
   328   222   331   236   325   354   231   360   326   219   218   324
   235   110   221   356   322   215   319   220   320   323   124   130
   364   117   140   134   127   113   133   142   318   118   144   131
   119   135   141   148   149   128   139   100    16   116   132   138
   112   145   143   121   126   316   115   122    22    23   129   226
   137   136   114   120   125   111   123   223    11   313   317   146
   147   241   217    65   239   216    31   361    14   238   310   242
   311   312   333   214   243   245   314    63   365   370   367   366
   368   371   213    68    64   210   379   374   373   377   372   386
   382   376   383   384   363   211   378   380    17   385    13   151


,x,y,VALUE
0,-853005.0,2785185.0,-9999
1,-852975.0,2785185.0,-9999
2,-852945.0,2785185.0,-9999
3,-852915.0,2785185.0,-9999
4,-852885.0,2785185.0,-9999


In [20]:
# Populate dataframe with default values
new_data = {
    'X' : map_df['x'],
    'Y' : map_df['y'],
    'EVCODE' : map_df['VALUE']
}
for c in continuous_formatted:
  new_data[c] = 0
for c in discrete_formatted:
  new_data[c] = False
new_df = pd.DataFrame(new_data)
new_df[ (new_df['X'] == -853005.0) & (new_df['Y'] == 2785185.0) ]


,X,Y,EVCODE,Tree_Cover,Shrub_Cover,Herb_Cover,Fill-NoData,Open_Water,Snow/Ice,Developed-Upland_Deciduous_Forest,...,Barren,Quarries-Strip_Mines-Gravel_Pits-Well_and_Wind_Pads,NASS-Vineyard,NASS-Row_Crop-Close_Grown_Crop,NASS-Row_Crop,NASS-Close_Grown_Crop,NASS-Wheat,NASS-Aquaculture,Cultivated_Crops,Sparse_Vegetation_Canopy
0,-853005.0,2785185.0,-9999,0,0,0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [18]:

for index, row in new_df.head().iterrows():
  code = row.iloc[2]
  key_table_row = key_table[key_table['VALUE'] == code].iloc[0]
  name = key_table_row.iloc[2]
  value = key_table_row.iloc[3]
  for
    new_df.at[index, '']
  display(row)


,0
X,-853005.0
Y,2785185.0
EVCODE,-9999
Tree_Cover,0
Shrub_Cover,0
Herb_Cover,0
Fill-NoData,False
Open_Water,False
Snow/Ice,False
Developed-Upland_Deciduous_Forest,False


,1
X,-852975.0
Y,2785185.0
EVCODE,-9999
Tree_Cover,0
Shrub_Cover,0
Herb_Cover,0
Fill-NoData,False
Open_Water,False
Snow/Ice,False
Developed-Upland_Deciduous_Forest,False


,2
X,-852945.0
Y,2785185.0
EVCODE,-9999
Tree_Cover,0
Shrub_Cover,0
Herb_Cover,0
Fill-NoData,False
Open_Water,False
Snow/Ice,False
Developed-Upland_Deciduous_Forest,False


,3
X,-852915.0
Y,2785185.0
EVCODE,-9999
Tree_Cover,0
Shrub_Cover,0
Herb_Cover,0
Fill-NoData,False
Open_Water,False
Snow/Ice,False
Developed-Upland_Deciduous_Forest,False


,4
X,-852885.0
Y,2785185.0
EVCODE,-9999
Tree_Cover,0
Shrub_Cover,0
Herb_Cover,0
Fill-NoData,False
Open_Water,False
Snow/Ice,False
Developed-Upland_Deciduous_Forest,False
